In [1]:
import pandas as pd
gold_second_futures = pd.read_csv('GC2026_second.csv')

In [ ]:
gold_second_futures.head()

In [3]:
import numpy as np
import pandas as pd
import keras

gold_second_futures['time'] = pd.to_datetime(
    gold_second_futures['time']
)

gold_second_futures['timestamp'] = (
    gold_second_futures['time'].astype('int64') // 10**9
)

X = gold_second_futures[
    ["timestamp", "volume"]
].to_numpy()

y = gold_second_futures[
    ["close"]
].to_numpy()   # (N, 1)

train_data = int(len(X) * 0.8)

X_train = X[:train_data]
y_train = y[:train_data]

X_test = X[train_data:]
y_test = y[train_data:]


sequence_length = 10
batch_size = 32

def create_sequences(X, y, sequence_length):
    X_sequences = []
    y_sequences = []

    for i in range(len(X) - sequence_length + 1):
        X_sequences.append(
            X[i:i + sequence_length]
        )

        y_sequences.append(
            y[i:i + sequence_length]
        )

    return (
        np.array(X_sequences),
        np.array(y_sequences)
    )


X_train_seq, y_train_seq = create_sequences(
    X_train,
    y_train,
    sequence_length
)

X_test_seq, y_test_seq = create_sequences(
    X_test,
    y_test,
    sequence_length
)

I0000 00:00:1788265936.773812   36595 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1788265936.843473   36595 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788265939.149968   36595 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [4]:
y_train_seq.shape

(2518391, 10, 1)

In [5]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.losses import MeanAbsoluteError
from tensorflow.keras.layers import SimpleRNN, Dense
import keras


# 1. Initialize the sequential model
model = Sequential()

model.add(keras.Input(shape=(10,2)))
# 2. Add the RNN layer 
# Input shape format: (time_steps, features)
model.add(SimpleRNN(units=64, return_sequences=True))

model.add(SimpleRNN(units=32, return_sequences=True))

# 3. Add a Dense output layer with linear activation for regression
model.add(Dense(units=1))

# 4. Compile the model
model.compile(optimizer='adam', loss=MeanAbsoluteError(), metrics=['mae'])

# View the network architecture
model.summary()

E0000 00:00:1788265946.166512   36595 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 10, 64)         │         4,288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 10, 32)         │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10, 1)          │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,425 (29.00 KB)

 Trainable params: 7,425 (29.00 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
model.fit(X_train_seq, y_train_seq, epochs=10, batch_size=32,shuffle=False, validation_data=(X_test_seq, y_test_seq))

Epoch 1/10


W0000 00:00:1788265951.602672   36595 cpu_allocator_impl.cc:82] Allocation of 201471280 exceeds 10% of free system memory.


78700/78700 ━━━━━━━━━━━━━━━━━━━━ 390s 5ms/step - loss: 3606.3591 - mae: 3606.3591 - val_loss: 1863.9927 - val_mae: 1863.9927
Epoch 2/10
78700/78700 ━━━━━━━━━━━━━━━━━━━━ 362s 5ms/step - loss: 1062.2906 - mae: 1062.2906 - val_loss: 261.9155 - val_mae: 261.9155
Epoch 3/10
78700/78700 ━━━━━━━━━━━━━━━━━━━━ 337s 4ms/step - loss: 120.4474 - mae: 120.4474 - val_loss: 261.9155 - val_mae: 261.9155
Epoch 4/10
78700/78700 ━━━━━━━━━━━━━━━━━━━━ 338s 4ms/step - loss: 120.4467 - mae: 120.4467 - val_loss: 261.9155 - val_mae: 261.9155
Epoch 5/10
78700/78700 ━━━━━━━━━━━━━━━━━━━━ 339s 4ms/step - loss: 120.4476 - mae: 120.4476 - val_loss: 261.9173 - val_mae: 261.9173
Epoch 6/10
78700/78700 ━━━━━━━━━━━━━━━━━━━━ 349s 4ms/step - loss: 120.4474 - mae: 120.4474 - val_loss: 261.9173 - val_mae: 261.9173
Epoch 7/10
78700/78700 ━━━━━━━━━━━━━━━━━━━━ 358s 5ms/step - loss: 120.4472 - mae: 120.4472 - val_loss: 261.9138 - val_mae: 261.9138
Epoch 8/10
78700/78700 ━━━━━━━━━━━━━━━━━━━━ 347s 4ms/step - loss: 120.4469 - mae:

In [7]:
model.evaluate(X_test_seq, y_test_seq)

19675/19675 ━━━━━━━━━━━━━━━━━━━━ 36s 2ms/step - loss: 261.9173 - mae: 261.9173


[261.91729736328125, 261.91729736328125]